In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import GPT2Tokenizer

BASE_DIR = r"C:\Users\MIND & MATTER\Desktop\ScratchLLM-95_conversational"
FINAL_MODEL_PATH = os.path.join(BASE_DIR, "checkpoints", "final_conversational_model.pt")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


In [2]:
class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len):
        super().__init__()
        inv_freq = 1.0 / (10000 ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)
        t = torch.arange(max_seq_len).float()
        freqs = torch.einsum("i,j->ij", t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos()[None, None, :, :])
        self.register_buffer("sin_cached", emb.sin()[None, None, :, :])

    def forward(self, x, seq_len):
        return self.cos_cached[:, :, :seq_len, :], self.sin_cached[:, :, :seq_len, :]

def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)

def apply_rope(q, k, cos, sin):
    q = (q * cos) + (rotate_half(q) * sin)
    k = (k * cos) + (rotate_half(k) * sin)
    return q, k

class SwiGLU(nn.Module):
    def __init__(self, dim, d_ff, dropout):
        super().__init__()
        self.w1 = nn.Linear(dim, d_ff, bias=False)
        self.w2 = nn.Linear(dim, d_ff, bias=False)
        self.w3 = nn.Linear(d_ff, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.w3(F.silu(self.w1(x)) * self.w2(x)))

class CausalSelfAttention(nn.Module):
    def __init__(self, dim, num_heads, max_seq_len, dropout):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.qkv_proj = nn.Linear(dim, dim * 3, bias=False)
        self.out_proj = nn.Linear(dim, dim, bias=False)
        self.rope = RotaryEmbedding(self.head_dim, max_seq_len)
        self.dropout = dropout

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv_proj(x).reshape(B, T, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        cos, sin = self.rope(x, T)
        q, k = apply_rope(q, k, cos, sin)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=self.dropout if self.training else 0.0)
        out = out.transpose(1, 2).reshape(B, T, C)
        return self.out_proj(out)

class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads, d_ff, max_seq_len, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(dim)
        self.attn = CausalSelfAttention(dim, num_heads, max_seq_len, dropout)
        self.ln2 = nn.LayerNorm(dim)
        self.ffn = SwiGLU(dim, d_ff, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

class ENGLlmModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.token_embed = nn.Embedding(config["vocab_size"], config["embed_dim"])
        self.blocks = nn.ModuleList([
            TransformerBlock(config["embed_dim"], config["num_heads"], config["d_ff"], config["max_seq_len"], config["dropout"])
            for _ in range(config["num_layers"])
        ])
        self.ln_f = nn.LayerNorm(config["embed_dim"])
        self.lm_head = nn.Linear(config["embed_dim"], config["vocab_size"], bias=False)

    def forward(self, input_ids):
        x = self.token_embed(input_ids)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        return self.lm_head(x)

print("Model architecture defined.")

Model architecture defined.


In [3]:
ckpt = torch.load(FINAL_MODEL_PATH, map_location=device, weights_only=False)
CONFIG = ckpt["config"]

model = ENGLlmModel(CONFIG)
model.load_state_dict(ckpt["model_state_dict"])
model.to(device)
model.eval()

print("Model loaded successfully.")
print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")

Model loaded successfully.
Total params: 114,391,040


In [4]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

In [5]:
@torch.no_grad()
def chat(prompt_text, max_new_tokens=120, repetition_penalty=1.3, min_len=40):
    model.eval()
    full_prompt = f"### Instruction:\n{prompt_text}\n\n### Response:\n"
    ids = tokenizer.encode(full_prompt)
    input_ids = torch.tensor([ids]).to(device)
    generated = list(ids)
    prompt_len = len(ids)

    for _ in range(max_new_tokens):
        input_ids_trunc = input_ids[:, -CONFIG["max_seq_len"]:]
        logits = model(input_ids_trunc)[:, -1, :]
        recent = generated[-128:]
        for tok_id in set(recent):
            logits[0, tok_id] /= repetition_penalty
        next_id = torch.argmax(logits, dim=-1, keepdim=True)
        input_ids = torch.cat([input_ids, next_id], dim=1)
        generated.append(next_id.item())

        decoded_new = tokenizer.decode(generated[prompt_len:])
        sentence_count = decoded_new.count(".") + decoded_new.count("!") + decoded_new.count("?")

        if next_id.item() == tokenizer.eos_token_id:
            break
        if len(generated) - prompt_len > min_len and sentence_count >= 2 and decoded_new.rstrip().endswith((".", "!", "?")):
            break
        if "### Instruction" in decoded_new[-30:]:
            break

    return tokenizer.decode(generated[prompt_len:], skip_special_tokens=True).strip()

print("chat() function ready.")

chat() function ready.


In [7]:
test_prompts = [
    "What should I do this weekend?",
    "Explain how photosynthesis works.",
    "Write a short poem about the ocean.",
    "Give me tips for staying productive.",
    "What's the best way to learn a new language?",
    "Recommend a good movie to watch tonight.",
    "How can I improve my sleep quality?",
    "Write a short story about a robot who wants to be human.",
    "What are some healthy breakfast ideas?",
    "How do I stay motivated to exercise?",
    "Explain the water cycle in simple terms.",
    "Give me advice for a first job interview.",
    "Write a birthday message for a close friend.",
    "What should I pack for a weekend camping trip?",
]

for p in test_prompts:
    print(f"### Instruction:\n{p}\n\n### Response:")
    print(chat(p))
    print("\n" + "-"*60 + "\n")

### Instruction:
What should I do this weekend?

### Response:
I would like to go on a vacation with my family. The weather is nice and sunny, so it can be a great time for me. It was an exciting experience that you had in mind when visiting the beach.

------------------------------------------------------------

### Instruction:
Explain how photosynthesis works.

### Response:
A cycle of photosynthesis is a process by which plants use light energy to convert carbon dioxide and water into chemical compounds, such as glucose or hydrogen peroxide (CO2uf5io4-L6O7opropylacosate. The organic molecules in the plant include oxygen, methane, potassium hydroxide, and other chemicals that are released during photosynthetic reactions.

------------------------------------------------------------

### Instruction:
Write a short poem about the ocean.

### Response:
The sea is filled with stars, rapping deep in the sky and its rays float so high that it can see as far away as the horizon.  It has b

In [13]:
your_prompt = "explain the water cycle in simple terms."  # change this
print(chat(your_prompt))

The water cycle is a natural process that occurs when sunlight passes through air droplets and causes precipitation, which can then be absorbed by rivers or lakes. The temperature of the atmosphere increases with altitude as it moves away from the ground due to rising sea levels caused by climate change.


In [14]:
# Base model, BEFORE apply_lora() is called
base_only = ENGLlmModel(CONFIG)
print(f"Base model params: {sum(p.numel() for p in base_only.parameters()):,}")

# Your final merged model
print(f"Merged model params: {sum(p.numel() for p in merged_model.parameters()):,}")

Base model params: 114,391,040


NameError: name 'merged_model' is not defined